# 3. Analysis data from Foot / HeartRate / GPX (location)
1. Read data and do basic analysis   

In [1]:
#basic
import os
import math
from pathlib import Path
from datetime import datetime # datetime

#analysis
import numpy as np
import pandas as pd
import folium # map
from IPython.display import IFrame
#Statistik
import scipy.stats as stats

#plot
import matplotlib.pyplot as plt
import seaborn as sns

1. Read data
We have 5 data files in `.csv` format:  
- One file corresponds to the date `2025/11/16`, surface type `Grass`, and subject `P06P`.  
- The other four files correspond to the date `2025/11/17`, surface types `Asphalt`, `Grass`, `Forest`, and `Sand`, and subject `S07`.

We will read these data files and load them into a single DataFrame.

In [2]:
# read data form
input_path = r"C:/Users/user/Desktop/Fatemeh/Analysis_Data/data/processed/Data frame from foot heart GPX"
# name of File as csv
os.makedirs(input_path, exist_ok=True)
file_names = [f for f in os.listdir(input_path) if f.endswith(".csv")]

name_of_dataframe = []
for i in range(len(file_names)):
    name_df = file_names[i]
    name = file_names[i].replace(".csv","")
    name_of_dataframe.append(name)
    path = f"{input_path}/{name_df}"
    data = pd.read_csv(path)
    globals()[name] = data

C:\Users\user\Oracel19c\admin\orcl\Temp\ipykernel_12492\4260495585.py:13: DtypeWarning: Columns (0,1) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(path)
C:\Users\user\Oracel19c\admin\orcl\Temp\ipykernel_12492\4260495585.py:13: DtypeWarning: Columns (0,1) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(path)
C:\Users\user\Oracel19c\admin\orcl\Temp\ipykernel_12492\4260495585.py:13: DtypeWarning: Columns (0,1) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(path)
C:\Users\user\Oracel19c\admin\orcl\Temp\ipykernel_12492\4260495585.py:13: DtypeWarning: Columns (0,1) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(path)
C:\Users\user\Oracel19c\admin\orcl\Temp\ipykernel_12492\4260495585.py:13: DtypeWarning: Columns (0,1) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.

In [3]:
name_of_dataframe

['Asphalt_S07_01_341_Date_2025_11_17_Time_10_09_36',
 'Forest_S07_02_492_Date_2025_11_17_Time_09_53_52',
 'Grass_P06P_01_845_Date_2025_11_16_Time_14_46_47',
 'Grass_S07_01_980_Date_2025_11_17_Time_09_27_24',
 'Sand__S07_01_073_Date_2025_11_17_Time_09_15_09']

# fillna
The DataFrames contain columns with `null` values due to timing mismatches:  
- Foot force data is sampled at **0.01-second** intervals.  
- Heart rate (`HR`) and location data (e.g., `latitude`, `longitude`, `elevation`) are sampled at **1-second** intervals.

To align the data, we fill the `null` values using the **`interpolate()`** method. This performs linear interpolation such that:
- If the first valid value is `109` and the next is `111`, the 99 intermediate values (corresponding to 0.01-second steps) are filled **in ascending order** from `109` to `111`.  
- Conversely, if values decrease (e.g., from `111` to `109`), the intermediate values are filled **in descending order**.

The interpolated columns include:  
`HR (bpm)`, `Speed (km/h)`, `Cadence`, `Distances (m)`, `latitude`, `longitude`, `elevation`.

In [ ]:
output_path = r"C:/Users/user/Desktop/Fatemeh/Analysis_Data/data/processed/Data frame from foot heart GPX fillna"
for i in name_of_dataframe:
    df = globals()[i].copy()
    df[df.columns[16:]] = df[df.columns[16:]].interpolate(method='polynomial', order=2)
    df['Speed (km/h)'] = df['Speed (km/h)'].abs()
    df['Cadence'] = df['Cadence'].abs()
    df['Distances (m)'] = df['Distances (m)'].abs()
    globals()[i] = df
    output_file = os.path.join(output_path, f"{i}.csv")
    globals()[i].to_csv(output_file, index=False) 



C:\Users\user\Oracel19c\admin\orcl\Temp\ipykernel_12492\2680927999.py:4: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  df[df.columns[16:]] = df[df.columns[16:]].interpolate(method='polynomial', order=2)
C:\Users\user\Oracel19c\admin\orcl\Temp\ipykernel_12492\2680927999.py:4: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  df[df.columns[16:]] = df[df.columns[16:]].interpolate(method='polynomial', order=2)
C:\Users\user\Oracel19c\admin\orcl\Temp\ipykernel_12492\2680927999.py:4: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  df[df.columns[16:]] = df[df.columns[16:]].interpolate(method='polynomial', order=2)
C:\Users\user